# AI Resume Screener - Evaluation & Analysis
This notebook demonstrates the dynamic NLP skill extraction, Semantic Expansion mapping, and Optimized TF-IDF Candidate Ranking.

In [ ]:
import sys
import os
import pandas as pd
import re
sys.path.append(os.path.abspath('../backend'))

from services.nlp_service import preprocess_text
from services.skill_extractor import extract_skills
from services.skill_expander import get_related_skills
from services.scoring_service import rank_candidates
from services.document_service import extract_text

job_description = "We are looking for a motivated undergraduate student to join our engineering team as a Software Development / AI Intern. The intern will assist in building, testing, and improving software applications that may involve backend services, web development, and emerging AI technologies. This role is designed for students who want hands-on experience in software development, problem solving, and modern development tools."

clean_jd = preprocess_text(job_description)
jd_skills = extract_skills(job_description)
print(f"Extracted Core Skills from JD: {jd_skills}")

expanded_demo = {}
for skill in jd_skills:
    related, is_broad = get_related_skills(skill)
    if is_broad:
        expanded_demo[skill] = related
print(f"\nBroad Expansions Found: {expanded_demo}\n")

resumes = []
data_dir = "../data"
for filename in os.listdir(data_dir):
    if filename.endswith('.docx') or filename.endswith('.pdf'):
        path = os.path.join(data_dir, filename)
        with open(path, 'rb') as f:
            raw_text = extract_text(f.read(), filename)
            
        raw_text_lower = raw_text.lower()
        matched = []
        missing = []
        
        for skill in jd_skills:
            escaped_skill = re.escape(skill)
            pattern = r'(?<![a-zA-Z0-9\-])' + escaped_skill + r'(?![a-zA-Z0-9\-])'
            has_exact = bool(re.search(pattern, raw_text_lower))
            
            related_skills, is_broad = get_related_skills(skill)
            if is_broad and related_skills:
                found_related = []
                for rs in related_skills:
                    rs_pattern = r'(?<![a-zA-Z0-9\-])' + re.escape(rs) + r'(?![a-zA-Z0-9\-])'
                    if re.search(rs_pattern, raw_text_lower):
                        found_related.append(rs)
                
                if has_exact or len(found_related) >= 1:
                    matched.append(skill)
                    matched.extend(found_related)
                else:
                    missing.append(skill)
            else:
                if has_exact:
                    matched.append(skill)
                else:
                    missing.append(skill)
        
        clean_text = preprocess_text(raw_text)
        
        resumes.append({
            "filename": filename,
            "text": clean_text,
            "matched_skills": sorted(list(set(matched))),
            "missing_skills": sorted(list(set(missing)))
        })

ranked = rank_candidates(jd_skills, resumes)

df = pd.DataFrame(ranked)
display(df[['rank', 'filename', 'similarity_score', 'matched_skills', 'missing_skills']])